In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# # ==============================
# # 1. Import Libraries
# # ==============================
# import cv2
# import numpy as np
# import matplotlib.pyplot as plt
# from skimage.filters import frangi
# import glob
# import os

# # ==============================
# # 2. Dataset Path
# # ==============================
# dataset_path = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset"
# # image_paths = sorted(glob.glob(os.path.join(dataset_path,"*.png")))[:50]
# image_paths = sorted(glob.glob(os.path.join(dataset_path,"*.png")))


# # Output folder
# save_folder = "/kaggle/working/vessel_maps"
# os.makedirs(save_folder,exist_ok=True)

# print("Total images:",len(image_paths))

# # ==============================
# # 3. Process Images
# # ==============================
# for path in image_paths:

#     # ---- Load image ----
#     img = cv2.imread(path)
#     img = cv2.resize(img,(512,512))

#     # ---- Green channel ----
#     green = img[:,:,1]

#     # ---- CLAHE (contrast enhance) ----
#     clahe = cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
#     green = clahe.apply(green)

#     # ---- Frangi Vesselness (Hessian based) ----
#     vessel = frangi(green,
#                     sigmas=range(1,6),
#                     black_ridges=True)

#     # ---- Normalize ----
#     vessel = cv2.normalize(vessel,None,0,255,cv2.NORM_MINMAX)
#     vessel = vessel.astype(np.uint8)

#     # ---- Invert (vessel black, background white) ----
#     vessel = 255 - vessel

#     # ---- Contrast boost ----
#     vessel = cv2.equalizeHist(vessel)

#     # ==========================
#     # 4. Plot
#     # ==========================
#     plt.figure(figsize=(8,4))

#     plt.subplot(1,2,1)
#     plt.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
#     plt.title("Original")
#     plt.axis("off")

#     plt.subplot(1,2,2)
#     plt.imshow(vessel,cmap="gray")
#     plt.title("Hessian Vessel Map")
#     plt.axis("off")

#     plt.show()

#     # ==========================
#     # 5. Save Output
#     # ==========================
#     name = os.path.basename(path)
#     cv2.imwrite(os.path.join(save_folder,name),vessel)

# print("Processing Complete")

In [ ]:
# ==============================
# 1. Import Libraries
# ==============================
import cv2
import numpy as np
from skimage.filters import frangi
import glob
import os

# ==============================
# 2. Dataset Path
# ==============================
dataset_path = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset"
image_paths = sorted(glob.glob(os.path.join(dataset_path,"*.png")))

# Output folder
save_folder = "/kaggle/working/vessel_maps"
os.makedirs(save_folder, exist_ok=True)

print("Total images:", len(image_paths))

# ==============================
# 3. Process Images
# ==============================
for path in image_paths:

    img = cv2.imread(path)
    img = cv2.resize(img,(512,512))

    # Green channel
    green = img[:,:,1]

    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    green = clahe.apply(green)

    # Frangi Vesselness
    vessel = frangi(
        green,
        sigmas=range(1,6),
        black_ridges=True
    )

    # Normalize
    vessel = cv2.normalize(vessel,None,0,255,cv2.NORM_MINMAX)
    vessel = vessel.astype(np.uint8)

    # Invert
    vessel = 255 - vessel

    # Contrast
    vessel = cv2.equalizeHist(vessel)

    # Save
    name = os.path.basename(path)
    cv2.imwrite(os.path.join(save_folder,name),vessel)

print("Processing Complete")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import glob
import os

dataset_path = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset"
vessel_path = "/kaggle/working/vessel_maps"

original_images = sorted(glob.glob(os.path.join(dataset_path,"*.png")))[:10]
vessel_images = sorted(glob.glob(os.path.join(vessel_path,"*.png")))[:10]

for orig, ves in zip(original_images, vessel_images):

    img = cv2.imread(orig)
    img = cv2.resize(img,(512,512))

    vessel = cv2.imread(ves,0)

    plt.figure(figsize=(8,4))

    plt.subplot(1,2,1)
    plt.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(vessel,cmap="gray")
    plt.title("Hessian Vessel Map")
    plt.axis("off")

    plt.show()

In [ ]:
!pip install timm

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import timm
import pandas as pd
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/amimulahasanrofik/abu-csv/data_info.csv")

features = ["age","gender","group"]
target = "thickness"

train_df, val_df = train_test_split(df,test_size=0.2,random_state=42)

In [ ]:
class FundusDataset(Dataset):

    def __init__(self,df,image_dir,vessel_dir):

        self.df=df
        self.image_dir=image_dir
        self.vessel_dir=vessel_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):

        row=self.df.iloc[idx]

        img_path=f"{self.image_dir}/{row.image_name}"
        vessel_path=f"{self.vessel_dir}/{row.image_name}"

        img=cv2.imread(img_path)
        img=cv2.resize(img,(224,224))
        img=img/255.0
        img=np.transpose(img,(2,0,1))

        vessel=cv2.imread(vessel_path,0)
        vessel=cv2.resize(vessel,(224,224))
        vessel=vessel/255.0
        vessel=np.expand_dims(vessel,0)

        tabular=row[features].values.astype(np.float32)

        label=row[target]

        return torch.tensor(img,dtype=torch.float32), \
               torch.tensor(vessel,dtype=torch.float32), \
               torch.tensor(tabular,dtype=torch.float32), \
               torch.tensor(label,dtype=torch.float32)

In [ ]:
train_dataset=FundusDataset(train_df,"/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset","/kaggle/working/vessel_maps")

train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)

In [ ]:
class TabularMLP(nn.Module):

    def __init__(self,input_dim):

        super().__init__()

        self.net=nn.Sequential(
            nn.Linear(input_dim,64),
            nn.ReLU(),
            nn.Linear(64,32)
        )

    def forward(self,x):

        return self.net(x)

In [ ]:
class HybridModel(nn.Module):

    def __init__(self,tabular_dim):

        super().__init__()

        # CNN
        self.cnn=models.resnet50(pretrained=True)
        self.cnn.fc=nn.Linear(2048,256)

        # Transformer
        self.vit=timm.create_model("vit_base_patch16_224",pretrained=True)
        self.vit.head=nn.Linear(768,256)

        # Tabular
        self.tabular=TabularMLP(tabular_dim)

        # Fusion
        self.fc=nn.Sequential(
            nn.Linear(256+256+32,128),
            nn.ReLU(),
            nn.Linear(128,1)
        )

    def forward(self,img,vessel,tab):

        f1=self.cnn(img)

        vessel=vessel.repeat(1,3,1,1)
        f2=self.vit(vessel)

        f3=self.tabular(tab)

        fused=torch.cat([f1,f2,f3],dim=1)

        out=self.fc(fused)

        return out

In [ ]:
model=HybridModel(tabular_dim=3).cuda()

criterion=nn.MSELoss()

optimizer=torch.optim.Adam(model.parameters(),lr=1e-4)

In [ ]:
for epoch in range(20):

    model.train()

    for img,vessel,tab,label in train_loader:

        img=img.cuda()
        vessel=vessel.cuda()
        tab=tab.cuda()
        label=label.cuda()

        pred=model(img,vessel,tab).squeeze()

        loss=criterion(pred,label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch:",epoch,"Loss:",loss.item())

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score